# T2D complications

Incident complications among people with prevalent diabetes diagnosed at age 18 or older:

* Chronic Kidney Disease
* Broad Renal Disease
* Heart Failure
* Foot Pathology
* Neuropathy
* Retinopathy

Each endpoint is fit as BASE, CLIN, and SOC (the marker model), with and without NMR. Clinical covariates and complication status and time come from `meta_clin.csv`. NMR features come from `assay`. `time_with_diab` is added to the base block.


In [ ]:
suppressPackageStartupMessages({
    library(ggpubr)
    library(patchwork)
    library(maplet)
    library(tidyverse)
    library(survival)
    library(glmnet)
    library(here)
})


# Load Data and Helpers


In [ ]:
# load tte helpers
source(here('helpers', 'prep_incident_time_to_event_df.r'))
source(here('helpers', 'summarize_time_to_event.r'))


In [ ]:
# load metadata and nmr data
meta <- read.csv(here('data', 'meta_clin.csv'), row.names = 1)
assay <- read.csv(here('data', 'assay.csv'), row.names = 1)
nmr_names <- assay %>% select(-StudyID) %>% names()


# Diabetes cohort

Restrict to prevalent diabetes with diagnosis age of at least 18 years, and record years already lived with diabetes at the sample. That duration enters the base model.


In [ ]:
meta <- meta %>%
  mutate(
    StudyID = as.character(StudyID),
    time_with_diab = age_at_sample - diab_prev_age
  ) %>%
  filter(diab_prev_age >= 18)

assay <- assay %>% mutate(StudyID = as.character(StudyID))

cat("Prevalent diabetes cohort (diagnosis age >= 18):", nrow(meta), "\n")
summary(meta %>% select(age_at_sample, diab_prev_age, time_with_diab))


# Configuration


In [ ]:
# source once
source(here('helpers', 'cv_glmnet_oof.r'))
source(here('helpers', 'fit_full_cox.r'))

cv_defaults <- list(
  outer_nfolds   = 5,
  inner_nfolds   = 10,
  nlambda        = 100,
  seed           = 1,
  standardize    = FALSE,
  parallel_outer = TRUE,
  alpha = 0
)

common_blocks <- list(
  base = c("age_at_sample", "sex", "time_with_diab"),
  clin = c("BMI_interp", "SBP_interp"),
  nmr  = nmr_names
)

common_scale_blocks <- list(
  base = c("age_at_sample", "time_with_diab"),
  clin = c("BMI_interp", "SBP_interp"),
  nmr  = nmr_names
)

model_specs <- list(
  base           = c("base"),
  base_nmr       = c("base", "nmr"),
  clin           = c("base", "clin"),
  clin_nmr       = c("base", "clin", "nmr"),
  marker         = c("base", "clin", "marker"),
  marker_nmr     = c("base", "clin", "marker", "nmr")
)


In [ ]:
resolve_blocks <- function(block_names, block_map) {
  missing_blocks <- setdiff(block_names, names(block_map))
  if (length(missing_blocks) > 0) {
    stop("Missing block definitions: ", paste(missing_blocks, collapse = ", "))
  }
  unique(unlist(block_map[block_names], use.names = FALSE))
}

make_penalty_factor <- function(predictors, unpenalized = character()) {
  pf <- rep(1, length(predictors))
  names(pf) <- predictors
  pf[intersect(unpenalized, predictors)] <- 0.1
  pf
}

make_model_maps <- function(marker_predictors,
                            marker_scale = character()) {
  list(
    blocks = c(common_blocks, list(marker = marker_predictors)),
    scale_blocks = c(common_scale_blocks, list(marker = marker_scale))
  )
}

get_required_vars <- function(maps,
                              specs = c("base", "clin", "marker"),
                              extra_vars = c("time", "status"),
                              exclude_blocks = c("nmr")) {
  used_blocks <- unique(unlist(model_specs[specs], use.names = FALSE))
  used_blocks <- setdiff(used_blocks, exclude_blocks)

  vars_needed <- c(
    extra_vars,
    resolve_blocks(used_blocks, maps$blocks)
  )

  unique(vars_needed)
}


filter_complete_cases_for_specs <- function(df, maps,
                                            specs = c("base", "clin", "marker"),
                                            extra_vars = c("time", "status"),
                                            exclude_blocks = c("nmr")) {
  vars_needed <- get_required_vars(
    maps = maps,
    specs = specs,
    extra_vars = extra_vars,
    exclude_blocks = exclude_blocks
  )

  df %>% filter(if_all(all_of(vars_needed), ~ !is.na(.)))
}

print_cv_summary <- function(results) {
  cat(sprintf("\nOOF C-index: %.4f (SE: %.4f)\n", results$c_index, results$se))
  cat(sprintf("95%% CI: [%.4f, %.4f]\n", results$ci_lower, results$ci_upper))
}

plot_cv_folds <- function(results) {
  n_folds <- length(results$fold_models)
  old_par <- par(no.readonly = TRUE)
  on.exit(par(old_par), add = TRUE)

  options(repr.plot.height = 2.5, repr.plot.width = 12)
  par(mfrow = c(1, n_folds))

  for (i in seq_len(n_folds)) {
    fold_model <- results$fold_models[[paste0("fold_", i)]]
    plot(fold_model, main = paste0("Fold\n", i))
  }
}

run_cv_spec <- function(df, spec_name, maps,
                        unpenalized = character(),
                        output_path = NULL,
                        plot_folds = FALSE,
                        model_engine = "glmnet",
                        fit_full = FALSE) { # optional full-data fit via fit_full_cox()

  if (!spec_name %in% names(model_specs)) {
    stop("Unknown spec_name: ", spec_name)
  }

  block_names   <- model_specs[[spec_name]]
  predictors    <- resolve_blocks(block_names, maps$blocks)
  vars_to_scale <- resolve_blocks(block_names, maps$scale_blocks)

  # fail fast if requested predictors are not in df
  missing_predictors <- setdiff(c("time", "status", predictors), names(df))
  if (length(missing_predictors) > 0) {
    stop(
      sprintf(
        "Spec '%s' requested variables not found in df: %s",
        spec_name,
        paste(missing_predictors, collapse = ", ")
      )
    )
  }

  # also verify vars_to_scale are present
  missing_scale <- setdiff(vars_to_scale, names(df))
  if (length(missing_scale) > 0) {
    stop(
      sprintf(
        "Spec '%s' vars_to_scale not found in df: %s",
        spec_name,
        paste(missing_scale, collapse = ", ")
      )
    )
  }

  # restrict df to only the model-relevant columns
  df_model <- df %>%
    dplyr::select(dplyr::all_of(c("StudyID", "time", "status", predictors))) %>%
    dplyr::filter(dplyr::if_all(dplyr::all_of(c("time", "status", predictors)), ~ !is.na(.)))

  unpenalized <- intersect(unpenalized, predictors)
  penalty.factor <- make_penalty_factor(predictors, unpenalized)

  args <- c(
    list(
      df = df_model,
      predictors = predictors,
      vars_to_scale = vars_to_scale,
      penalty.factor = penalty.factor,
      model_engine = model_engine
    ),
    cv_defaults
  )
  results <- do.call(cv_glmnet_cox_oof, args)
  print_cv_summary(results)

  # fit one model on all complete cases and store as results$full_model
  if (fit_full) {
    full_args <- c(
      list(
        df             = df_model,
        predictors     = predictors,
        vars_to_scale  = vars_to_scale,
        penalty.factor = penalty.factor,
        model_engine   = model_engine
      ),
      cv_defaults[setdiff(names(cv_defaults), c("outer_nfolds", "parallel_outer"))]
    )
    results$full_model <- do.call(fit_full_cox, full_args)
  }

  if (plot_folds && model_engine == "glmnet") {
    plot_cv_folds(results)
  }
  if (!is.null(output_path)) {
    saveRDS(results, output_path)
  }
  invisible(results)
}

inspect_spec <- function(df, spec_name, maps) {
  block_names <- model_specs[[spec_name]]
  predictors  <- resolve_blocks(block_names, maps$blocks)

  list(
    spec_name = spec_name,
    n_predictors = length(predictors),
    first_predictors = head(predictors, 10),
    missing_in_df = setdiff(predictors, names(df))
  )
}


## Diabetes complications map

One marker block is shared by every T2D complication. HDL uses the `meta_clin` column names.


In [ ]:
# Diabetes complications marker map
# Scale continuous observed values and time-delay terms.
# Do NOT scale M_* indicators or binary treatment flags.
diabetes_complications_maps <- make_model_maps(
  marker_predictors = c(
    # Glycemic burden
    "M_a1c",
    "A1C_obs",
    "T_a1c",

    # A1C variability
    "M_a1c_var",
    "A1C_cv_obs",
    "T_a1c_var",

    # Kidney function and trajectory
    "M_egfr",
    "EGFR_obs",
    "T_egfr",
    "M_egfr_slope",
    "EGFR_slope_obs",
    "T_egfr_slope",

    # BP treatment intensity; SBP itself is already in the clin block
    "bp_treatment",
    "statin_treatment",

    # Lipids
    "M_ldl",
    "LDL_obs",
    "T_ldl",
    "M_hdl_choles",
    "HDL_CHOLES_obs",
    "T_hdl_choles",
    "M_triglycerides",
    "TRIGLYCERIDES_obs",
    "T_triglycerides",

    # Albumin
    "M_albumin",
    "ALBUMIN_obs",
    "T_albumin",

    # Smoking
    "M_smoke",
    "smk_current",
    "smk_former"
  ),

  marker_scale = c(
    "A1C_obs",
    "T_a1c",
    "A1C_cv_obs",
    "T_a1c_var",
    "EGFR_obs",
    "T_egfr",
    "EGFR_slope_obs",
    "T_egfr_slope",
    "LDL_obs",
    "T_ldl",
    "HDL_CHOLES_obs",
    "T_hdl_choles",
    "TRIGLYCERIDES_obs",
    "T_triglycerides",
    "ALBUMIN_obs",
    "T_albumin"
  )
)


# Endpoints

Chronic kidney disease uses `ckd_status` and `ckd_time` from `meta_clin.csv`. Heart failure uses `heart_failure_status` and `heart_failure_time`, also from `meta_clin.csv`, not `hf_cm_status`. The microvascular outcomes are also on `meta_clin.csv`.

Model objects are written under `outputs/secondary_diabetes/`. Summary tables go to `outputs/tables/` and figures to `outputs/figures/`.


In [ ]:
DIABETES_OOF_ROOT <- here('outputs/secondary_diabetes')
DIABETES_SUMMARY_ROOT <- here('outputs/tables')
DIABETES_FIGURE_ROOT <- here('outputs/figures')
dir.create(file.path(DIABETES_OOF_ROOT, 'absolute_risks'), recursive = TRUE, showWarnings = FALSE)
dir.create(DIABETES_SUMMARY_ROOT, recursive = TRUE, showWarnings = FALSE)
dir.create(DIABETES_FIGURE_ROOT, recursive = TRUE, showWarnings = FALSE)

# plot_label is the text after "T2D ->" on the C-index panel.
# delta_label is the y-axis label on the delta-C figure.
# legend is collected on the last panel of the stacked C-index figure.
DIABETES_ENDPOINTS <- list(
  ckd = list(
    tag = 't2d_ckd',
    status_col = 'ckd_status',
    time_col = 'ckd_time',
    plot_label = 'Chronic Kidney Disease',
    delta_label = 'T2D Chronic Kidney Disease',
    legend = 'none'
  ),
  nephropathy = list(
    tag = 't2d_nephropathy',
    status_col = 't2d_kidney_status',
    time_col = 't2d_kidney_time',
    plot_label = 'Broad Renal Disease',
    delta_label = 'T2D Broad Renal Disease',
    legend = 'none'
  ),
  hf = list(
    tag = 't2d_heart_failure',
    status_col = 'heart_failure_status',
    time_col = 'heart_failure_time',
    plot_label = 'Heart Failure',
    delta_label = 'T2D Heart Failure',
    legend = 'none'
  ),
  foot = list(
    tag = 't2d_foot',
    status_col = 't2d_foot_status',
    time_col = 't2d_foot_time',
    plot_label = 'Foot Pathology',
    delta_label = 'T2D Foot Pathology',
    legend = 'none'
  ),
  neuropathy = list(
    tag = 't2d_neuropathy',
    status_col = 't2d_nerve_status',
    time_col = 't2d_nerve_time',
    plot_label = 'Neuropathy',
    delta_label = 'T2D Neuropathy',
    legend = 'none'
  ),
  retinopathy = list(
    tag = 't2d_retinopathy',
    status_col = 't2d_retinop_status',
    time_col = 't2d_retinop_time',
    plot_label = 'Retinopathy',
    delta_label = 'T2D Retinopathy',
    legend = 'bottom'
  )
)

DIABETES_SPECS <- names(model_specs)


In [ ]:
required_cols <- unique(c(
  unlist(diabetes_complications_maps$blocks[c('base', 'clin', 'marker')], use.names = FALSE),
  vapply(DIABETES_ENDPOINTS, function(cfg) cfg$status_col, character(1)),
  vapply(DIABETES_ENDPOINTS, function(cfg) cfg$time_col, character(1))
))
missing_cols <- setdiff(required_cols, names(meta))
if (length(missing_cols) > 0) {
  stop('Missing columns in the cohort: ', paste(missing_cols, collapse = ', '))
}


In [ ]:
source(here('helpers', 'fold_specific_oof_absolute_risk.r'))

run_diabetes_endpoint <- function(cfg, maps = diabetes_complications_maps) {
  message('\n==== ', cfg$delta_label, ' ====')

  meta_use <- prep_incident_time_to_event_df(
    meta,
    status_col = cfg$status_col,
    time_col = cfg$time_col,
    time_filter = paste(cfg$time_col, '> 0')
  )
  print(summarize_time_to_event(meta_use))

  meta_use <- filter_complete_cases_for_specs(
    df = meta_use,
    maps = maps,
    specs = c('base', 'clin', 'marker')
  )
  meta_use_nmr <- meta_use %>% inner_join(assay, by = 'StudyID')

  cat('meta_use dimensions:', dim(meta_use), '\n')
  cat('meta_use_nmr dimensions:', dim(meta_use_nmr), '\n')
  tte_analytical <- summarize_time_to_event(meta_use_nmr) %>%
    mutate(endpoint = cfg$delta_label, tag = cfg$tag)
  print(tte_analytical)

  for (spec_name in DIABETES_SPECS) {
    results <- run_cv_spec(
      df = meta_use_nmr,
      spec_name = spec_name,
      maps = maps,
      unpenalized = c('age_at_sample', 'sex'),
      model_engine = 'glmnet',
      output_path = file.path(DIABETES_OOF_ROOT, paste0(cfg$tag, '_', spec_name, '_oof.rds')),
      fit_full = spec_name %in% c('marker', 'marker_nmr')
    )
    print(results$fold_info)
    print(summary(results$oof_predictions$lp))
    rm(results)
    gc(verbose = FALSE)
  }

  for (spec_name in c('marker', 'marker_nmr')) {
    saved <- readRDS(file.path(DIABETES_OOF_ROOT, paste0(cfg$tag, '_', spec_name, '_oof.rds')))
    oof_risk <- compute_fold_specific_oof_absolute_risk(
      results = saved,
      df = meta_use_nmr,
      vars_to_scale = resolve_blocks(model_specs[[spec_name]], maps$scale_blocks),
      prediction_times = c(1, 3, 5)
    )
    saveRDS(
      oof_risk,
      file.path(
        DIABETES_OOF_ROOT, 'absolute_risks',
        paste0(cfg$tag, '_', spec_name, '_fold_baseline_risk.rds')
      )
    )
    rm(saved, oof_risk)
    gc(verbose = FALSE)
  }

  rm(meta_use, meta_use_nmr)
  tte_analytical
}


In [ ]:
ckd_tte <- run_diabetes_endpoint(DIABETES_ENDPOINTS$ckd)
nephropathy_tte <- run_diabetes_endpoint(DIABETES_ENDPOINTS$nephropathy)
hf_tte <- run_diabetes_endpoint(DIABETES_ENDPOINTS$hf)
foot_tte <- run_diabetes_endpoint(DIABETES_ENDPOINTS$foot)
neuropathy_tte <- run_diabetes_endpoint(DIABETES_ENDPOINTS$neuropathy)
retinopathy_tte <- run_diabetes_endpoint(DIABETES_ENDPOINTS$retinopathy)

diabetes_tte <- bind_rows(ckd_tte, nephropathy_tte, hf_tte, foot_tte, neuropathy_tte, retinopathy_tte)
diabetes_tte


# Absolute C-index

C-index tables are read back from the saved out-of-fold fits.


In [ ]:
source(here('helpers', 'plot_oof_cindex_primary.r'))

PLOT_MARKER_LABEL <- 'SOC+'
PLOT_TYPE_LABELS <- c('SOC' = 'No NMR', 'NMR' = 'NMR')

DIABETES_SPEC_MAP <- data.frame(
  spec  = c('base', 'base_nmr', 'clin', 'clin_nmr', 'marker', 'marker_nmr'),
  model = c('base', 'base', 'clin', 'clin', 'marker', 'marker'),
  type  = c('SOC', 'NMR', 'SOC', 'NMR', 'SOC', 'NMR'),
  stringsAsFactors = FALSE
)

diabetes_oof_path <- function(tag, spec) {
  file.path(DIABETES_OOF_ROOT, paste0(tag, '_', spec, '_oof.rds'))
}

write_diabetes_cindex_csv <- function(cfg) {
  rows <- lapply(seq_len(nrow(DIABETES_SPEC_MAP)), function(i) {
    spec <- DIABETES_SPEC_MAP$spec[[i]]
    path <- diabetes_oof_path(cfg$tag, spec)
    if (!file.exists(path)) stop('Missing OOF RDS: ', path)
    res <- readRDS(path)
    data.frame(
      model    = DIABETES_SPEC_MAP$model[[i]],
      type     = DIABETES_SPEC_MAP$type[[i]],
      c_index  = res$c_index,
      ci_lower = res$ci_lower,
      ci_upper = res$ci_upper,
      stringsAsFactors = FALSE
    )
  })

  out <- dplyr::bind_rows(rows)
  path <- file.path(DIABETES_SUMMARY_ROOT, paste0('oof_', cfg$tag, '_cindex_ci.csv'))
  write.csv(out, path, row.names = FALSE)
  message('Wrote ', path)
  out
}

diabetes_cindex_list <- lapply(DIABETES_ENDPOINTS, write_diabetes_cindex_csv)

diabetes_cindex_summary <- dplyr::bind_rows(lapply(names(DIABETES_ENDPOINTS), function(nm) {
  cfg <- DIABETES_ENDPOINTS[[nm]]
  diabetes_cindex_list[[nm]] %>%
    mutate(endpoint = cfg$delta_label, tag = cfg$tag, .before = model)
}))

write.csv(
  diabetes_cindex_summary,
  file.path(DIABETES_SUMMARY_ROOT, 'oof_t2d_cindex_summary.csv'),
  row.names = FALSE
)

diabetes_cindex_summary


In [ ]:
diabetes_title <- function(label) {
  as.expression(bquote(plain(T2D) %->% ~ .(label)))
}

diabetes_plots <- lapply(names(DIABETES_ENDPOINTS), function(nm) {
  cfg <- DIABETES_ENDPOINTS[[nm]]
  plot_oof_cindex(
    df = diabetes_cindex_list[[nm]],
    title = diabetes_title(cfg$plot_label),
    xlim = c(0.5, 0.8),
    marker_label = PLOT_MARKER_LABEL,
    type_labels = PLOT_TYPE_LABELS
  ) +
    theme(legend.position = cfg$legend)
})
names(diabetes_plots) <- names(DIABETES_ENDPOINTS)

options(repr.plot.height = 7.5, repr.plot.width = 3)

p_all_diabetes <- diabetes_plots$ckd / diabetes_plots$nephropathy / diabetes_plots$hf / diabetes_plots$foot / diabetes_plots$neuropathy / diabetes_plots$retinopathy
p_all_diabetes

ggsave(
  file.path(DIABETES_FIGURE_ROOT, 'DIABETES_A.png'),
  plot = p_all_diabetes,
  width = 3,
  height = 7.5,
  dpi = 600
)


# Delta C-index

Paired comparisons of the saved out-of-fold fits. Benjamini-Hochberg adjustment is applied within each comparison (BASE, CLIN, SOC+, and CLIN+NMR vs SOC+) across the six endpoints in this panel. 


In [ ]:
source(here('helpers', 'delta_cindex_oof.r'))

DELTA_SPEC_MAP <- c(
  base       = 'base_SOC',
  base_nmr   = 'base_NMR',
  clin       = 'clin_SOC',
  clin_nmr   = 'clin_NMR',
  marker     = 'marker_SOC',
  marker_nmr = 'marker_NMR'
)

DELTA_COMPARISONS <- data.frame(
  model_1 = c('base_SOC',   'clin_SOC',   'marker_SOC', 'marker_SOC'),
  model_2 = c('base_NMR',   'clin_NMR',   'marker_NMR', 'clin_NMR'),
  stringsAsFactors = FALSE
)

load_diabetes_oof_results <- function(tag) {
  results <- lapply(names(DELTA_SPEC_MAP), function(sp) {
    path <- diabetes_oof_path(tag, sp)
    if (!file.exists(path)) stop('Missing OOF RDS: ', path)
    strip_oof_result(readRDS(path))
  })
  names(results) <- unname(DELTA_SPEC_MAP)
  results
}

run_diabetes_delta <- function(cfg, parallel = TRUE, workers = 2L) {
  message('Delta C: ', cfg$delta_label)
  results <- load_diabetes_oof_results(cfg$tag)
  delta <- run_delta_cindex_batch(
    results = results,
    comparisons = DELTA_COMPARISONS,
    strip_first = TRUE,
    parallel = parallel,
    workers = workers,
    parallel_transport = 'rds',
    close_workers_on_exit = TRUE
  )
  delta$endpoint <- cfg$delta_label
  delta$tag <- cfg$tag
  path <- file.path(DIABETES_SUMMARY_ROOT, paste0('oof_', cfg$tag, '_delta_cindex.csv'))
  write.csv(delta, path, row.names = FALSE)
  message('Wrote ', path)
  rm(results)
  invisible(gc(verbose = FALSE))
  delta
}

delta_all_diabetes <- bind_rows(lapply(DIABETES_ENDPOINTS, run_diabetes_delta)) %>%
  mutate(
    boost = case_when(
      model_1 == 'base_SOC'   & model_2 == 'base_NMR'   ~ 'BASE',
      model_1 == 'clin_SOC'   & model_2 == 'clin_NMR'   ~ 'CLIN',
      model_1 == 'marker_SOC' & model_2 == 'marker_NMR' ~ 'SOC+',
      model_1 == 'marker_SOC' & model_2 == 'clin_NMR'   ~ 'CLIN+NMR vs SOC+',
      TRUE ~ NA_character_
    )
  ) %>%
  filter(!is.na(boost)) %>%
  group_by(boost) %>%
  mutate(FDR = p.adjust(p_value, method = 'BH')) %>%
  ungroup()

write.csv(
  delta_all_diabetes,
  file.path(DIABETES_SUMMARY_ROOT, 'oof_t2d_delta_cindex_summary.csv'),
  row.names = FALSE
)

delta_all_diabetes


In [ ]:
library(dplyr)
library(ggplot2)
library(ggh4x)
library(patchwork)
library(here)

plot_delta_c_boost <- function(
    df,
    endpoint_order,
    x_by = 0.05,
    x_max = 0.15,
    x_min_comparison = -0.05,
    x_min_soc = -0.01,
    annotation_pad = 0.018
) {

  boost_order <- c(
    "BASE",
    "CLIN",
    "SOC+",
    "CLIN+NMR vs SOC+"
  )

  # --------------------------------------------------------------------------
  # Prepare data
  # --------------------------------------------------------------------------

  plot_df <- df %>%
    mutate(
      boost = case_when(
        model_1 == "base_SOC"   & model_2 == "base_NMR"   ~ "BASE",
        model_1 == "clin_SOC"   & model_2 == "clin_NMR"   ~ "CLIN",
        model_1 == "marker_SOC" & model_2 == "marker_NMR" ~ "SOC+",
        model_1 == "marker_SOC" & model_2 == "clin_NMR"   ~ "CLIN+NMR vs SOC+",
        TRUE ~ NA_character_
      ),

      boost = factor(
        boost,
        levels = boost_order
      ),

      endpoint = factor(
        endpoint,
        levels = rev(endpoint_order)
      )
    ) %>%
    filter(!is.na(boost))


  # --------------------------------------------------------------------------
  # Add explicit positions for significance labels
  # --------------------------------------------------------------------------
  #
  # This is preferable to relying on large +/- hjust values right at the
  # panel boundary.
  #

  plot_df <- plot_df %>%
    mutate(
      sig_x = if_else(
        delta_c >= 0,
        ci_upper + 0.004,
        ci_lower - 0.004
      ),

      sig_hjust = if_else(
        delta_c >= 0,
        0,
        1
      ),

      sig_label = case_when(
        is.na(FDR)     ~ NA_character_,
        FDR >= 0.05    ~ "n.s.",
        FDR < 0.001    ~ "***",
        FDR < 0.01     ~ "**",
        TRUE           ~ "*"
      )
    )


  # --------------------------------------------------------------------------
  # Plot
  # --------------------------------------------------------------------------

  g <- ggplot(
    plot_df,
    aes(
      x = delta_c,
      y = endpoint
    )
  ) +

    # Bars
    geom_col(
      width = 0.62,
      fill = "steelblue"
    ) +

    # Confidence intervals
    geom_errorbarh(
      aes(
        xmin = ci_lower,
        xmax = ci_upper
      ),
      height = 0.18,
      linewidth = 0.45
    ) +

    # Significance labels
    geom_text(
      data = plot_df %>%
        filter(!is.na(sig_label)),
      aes(
        x = sig_x,
        label = sig_label,
        hjust = sig_hjust
      ),
      size = 2.7,
      vjust = 0.7
    ) +

    # Zero reference
    geom_vline(
      xintercept = 0,
      linewidth = 0.25,
      colour = "black"
    ) +

    # Free scale required for panel-specific x scales
    facet_wrap(
      ~ boost,
      nrow = 1,
      scales = "free_x"
    ) +

    # ------------------------------------------------------------------------
    # Facet-specific x scales
    #
    # Important:
    #
    # First 3 panels DISPLAY ticks only through 0.15,
    # but their actual scale extends to 0.168 to leave room for stars.
    #
    # Last panel extends slightly beyond -0.05 and +0.15 for annotation.
    # ------------------------------------------------------------------------

    ggh4x::facetted_pos_scales(
  x = list(

    # BASE / CLIN
    boost %in% c("BASE", "CLIN") ~
      scale_x_continuous(
        limits = c(
          0,
          x_max + annotation_pad
        ),

        breaks = seq(
          0,
          x_max,
          by = x_by
        ),

        labels = function(x) {
          dplyr::case_when(
            x == 0 ~ "0",
            TRUE ~ sub("^0", "", sprintf("%.2f", x))
          )
        },

        expand = expansion(
          mult = c(0, 0)
        )
      ),

    # SOC+ -- slight negative space so CIs do not clip
    boost == "SOC+" ~
      scale_x_continuous(
        limits = c(
          -0.01,
          x_max + annotation_pad
        ),

        breaks = seq(
          0,
          x_max,
          by = x_by
        ),

        labels = function(x) {
          dplyr::case_when(
            x == 0 ~ "0",
            TRUE ~ sub("^0", "", sprintf("%.2f", x))
          )
        },

        expand = expansion(
          mult = c(0, 0)
        )
      ),

    # CLIN+NMR vs SOC+
    boost == "CLIN+NMR vs SOC+" ~
      scale_x_continuous(
        limits = c(
          x_min_comparison - 0.012,
          x_max + annotation_pad
        ),

        breaks = seq(
          x_min_comparison,
          x_max,
          by = x_by
        ),

        labels = function(x) {
          dplyr::case_when(
            x == 0 ~ "0",
            x < 0 ~ paste0(
              "\u2212",
              sub(
                "^0",
                "",
                sprintf("%.2f", abs(x))
              )
            ),
            TRUE ~ sub(
              "^0",
              "",
              sprintf("%.2f", x)
            )
          )
        },

        expand = expansion(
          mult = c(0, 0)
        )
      )
  )
) +

    scale_y_discrete(
      drop = FALSE,
      expand = expansion(
        add = c(0.50, 0.55)
      )
    ) +

    labs(
      x = expression(Delta~"C-index (95% CI)"),
      y = NULL,
      title = NULL
    ) +

    theme_bw(
      base_size = 11
    ) +

    theme(

      # ------------------------------------------------------------
      # Facet headers
      # ------------------------------------------------------------

      strip.background = element_rect(
        fill = "grey92",
        colour = NA
      ),

      strip.text = element_text(
        face = "plain",
        size = 9.5,
        margin = margin(
          t = 4,
          r = 2,
          b = 4,
          l = 2
        )
      ),

      # ------------------------------------------------------------
      # Panels
      # ------------------------------------------------------------

      panel.border = element_blank(),

      panel.grid.major.y = element_blank(),

      panel.grid.major.x = element_line(
        colour = "grey90",
        linewidth = 0.3
      ),

      panel.grid.minor = element_blank(),

      # More space between facets
      panel.spacing.x = grid::unit(
        0.35,
        "cm"
      ),

      # ------------------------------------------------------------
      # Axes
      # ------------------------------------------------------------

      axis.line.x = element_line(
        colour = "black",
        linewidth = 0.4
      ),

      axis.ticks.x = element_line(
        colour = "black",
        linewidth = 0.35
      ),

      axis.ticks.y = element_blank(),

      axis.text.x = element_text(
        size = 8,
        colour = "black",
        margin = margin(t = 2)
      ),

      axis.text.y = element_text(
        size = 8.7,
        colour = "black",
        margin = margin(r = 3)
      ),

      axis.title.x = element_text(
        size = 10.5,
        margin = margin(t = 5)
      ),

      # ------------------------------------------------------------
      # Plot title
      # ------------------------------------------------------------

      plot.title = element_text(
        hjust = 0.5,
        face = "plain",
        size = 12,
        margin = margin(
          b = 6
        )
      ),

      # ------------------------------------------------------------
      # Margins
      # ------------------------------------------------------------

      plot.margin = margin(
        t = 3,
        r = 5,
        b = 2,
        l = 3
      )
    )

  return(g)
}


In [ ]:
options(repr.plot.height = 1.75, repr.plot.width = 7.75)

diabetes_order <- c(
  'T2D Chronic Kidney Disease',
  'T2D Broad Renal Disease',
  'T2D Heart Failure',
  'T2D Foot Pathology',
  'T2D Neuropathy',
  'T2D Retinopathy'
)

p_all_diabetes_delta <- plot_delta_c_boost(
  delta_all_diabetes,
  endpoint_order = diabetes_order,
  x_by = 0.05,
  x_max = 0.15
) +
  labs(title = 'Incremental discrimination with NMR for T2D Progression')

p_all_diabetes_delta

ggsave(
  file.path(DIABETES_FIGURE_ROOT, 'DIABETES_B.png'),
  plot = p_all_diabetes_delta,
  width = 7.75,
  height = 1.75,
  dpi = 600
)
